# Speaker Clustering — External Dataset (Kaggle)

Self-contained Kaggle version of `speaker_clustering/check_filename_pattern.py` + `cluster_external_dataset.py`.

Pipeline: check the `{chapter}_{index}.ext` filename pattern on `audio_file` → derive `chapter`/`duration` for every row → embed one representative segment per chapter (SpeechBrain ECAPA-TDNN) → cluster chapters into `speaker_id` → apply it dataset-wide → rename `audio_file`→`audio` → push back to the Hub.

**Before running the push cell at the end**, read the per-speaker duration table it prints and sanity-check a few chapters per speaker by ear — the clustering threshold below is an untuned starting point, not a validated value.

**Enable a GPU accelerator** in the notebook settings (Settings → Accelerator) for much faster embedding extraction — this notebook auto-detects and uses it if available.

In [ ]:
!pip install -q speechbrain

In [ ]:
import re
import sys
from collections import Counter, defaultdict

import numpy as np
import torch
import librosa
from sklearn.cluster import AgglomerativeClustering
from tqdm.auto import tqdm
from datasets import load_dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# ---- Config: edit these ----
REPO_ID = "Minervus00/moore_audio_bible"
SPLIT = "train"

# Prefer Kaggle Secrets over pasting a raw token here:
#   from kaggle_secrets import UserSecretsClient
#   HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
HF_TOKEN = "hf_xxxxxxxxxxxx"

AUDIO_COLUMN = "audio_file"
TEXT_COLUMN = "text"

DISTANCE_THRESHOLD = 0.35   # lower = more/smaller speaker clusters. Ignored if N_CLUSTERS is set.
N_CLUSTERS = None           # set an int to force an exact speaker count instead

PUSH_TO_HUB = False          # flip to True once you've sanity-checked the clustering
OUTPUT_REPO_ID = None        # e.g. "Minervus00/moore_audio_bible" to overwrite, or a different repo id

FILENAME_RE = re.compile(r"^(?P<chapter>.+)_(?P<index>\d+)\.[A-Za-z0-9]+$")

## Step 1 — Load the dataset and check the filename pattern
Checks every row, not just one example, before we trust the inferred chapter grouping.

In [ ]:
print(f"Loading {REPO_ID} (split={SPLIT})...")
dataset = load_dataset(REPO_ID, split=SPLIT, token=HF_TOKEN)
print(f"{len(dataset)} rows")

if AUDIO_COLUMN not in dataset.column_names:
    raise SystemExit(f"Column '{AUDIO_COLUMN}' not found. Available columns: {dataset.column_names}")

# dataset[AUDIO_COLUMN] returns path/array/sampling_rate dicts; accessing just
# "path" via this bulk column access does not decode audio.
paths = dataset[AUDIO_COLUMN]

matched = 0
unmatched_examples = []
chapter_segment_counts = defaultdict(int)
chapter_index_sets = defaultdict(set)
duplicate_index_examples = []

for i, entry in enumerate(paths):
    path = entry["path"] if isinstance(entry, dict) else entry
    m = FILENAME_RE.match(path)
    if not m:
        unmatched_examples.append((i, path))
        continue
    matched += 1
    chapter = m.group("chapter")
    index = m.group("index")
    chapter_segment_counts[chapter] += 1
    if index in chapter_index_sets[chapter]:
        duplicate_index_examples.append((chapter, index, path))
    chapter_index_sets[chapter].add(index)

total = len(paths)
print(f"\nMatched pattern: {matched}/{total} ({100 * matched / total:.1f}%)")
print(f"Distinct chapters inferred: {len(chapter_segment_counts)}")
if chapter_segment_counts:
    counts = sorted(chapter_segment_counts.values())
    print(f"Segments per chapter: min={counts[0]}, max={counts[-1]}, median={counts[len(counts) // 2]}")
if unmatched_examples:
    print(f"\n{len(unmatched_examples)} filenames did NOT match the pattern, e.g.:")
    for i, path in unmatched_examples[:10]:
        print(f"  row {i}: {path!r}")
if duplicate_index_examples:
    print(f"\n{len(duplicate_index_examples)} duplicate (chapter, index) pairs found, e.g.:")
    for chapter, index, path in duplicate_index_examples[:10]:
        print(f"  chapter={chapter!r} index={index!r} path={path!r}")
if not unmatched_examples and not duplicate_index_examples:
    print("\nPattern holds cleanly across the whole dataset.")

**Only continue past this point if the pattern check above looks clean** (near-100% match rate, no unexpected duplicate indices). If it doesn't, stop and adjust `FILENAME_RE` to match what you're actually seeing before proceeding.

## Step 2 — Derive `chapter` and `duration` for every row
This decodes every row's audio once (unavoidable — `duration` needs to exist for the whole dataset, not just representative rows).

In [ ]:
def add_chapter_and_duration(example):
    path = example[AUDIO_COLUMN]["path"]
    m = FILENAME_RE.match(path)
    if not m:
        raise ValueError(f"Filename does not match the expected pattern: {path!r}")
    array = example[AUDIO_COLUMN]["array"]
    sr = example[AUDIO_COLUMN]["sampling_rate"]
    # Return only the new columns, not the whole `example` dict: handing back
    # the audio column's value forces datasets to re-encode it from the
    # decoded array, which drops the original "path".
    return {"chapter": m.group("chapter"), "duration": len(array) / sr}

dataset = dataset.map(add_chapter_and_duration, desc="chapter/duration")
print(dataset[0])

## Step 3 — Extract one speaker embedding per chapter
Assumes a single narrator per chapter, so only the longest segment of each chapter is embedded.

In [ ]:
best_idx_by_chapter = {}
best_duration_by_chapter = {}
for i, (chapter, duration) in enumerate(zip(dataset["chapter"], dataset["duration"])):
    if chapter not in best_duration_by_chapter or duration > best_duration_by_chapter[chapter]:
        best_duration_by_chapter[chapter] = duration
        best_idx_by_chapter[chapter] = i

print(f"{len(best_idx_by_chapter)} chapters found.")

In [ ]:
from speechbrain.inference.speaker import EncoderClassifier

MODEL_SAMPLE_RATE = 16000

print("Loading speaker embedding model (speechbrain/spkrec-ecapa-voxceleb)...")
classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="model_cache",
    run_opts={"device": DEVICE},
)

def embed_waveform(waveform, sample_rate):
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=-1)
    waveform = waveform.astype(np.float32)
    if sample_rate != MODEL_SAMPLE_RATE:
        waveform = librosa.resample(waveform, orig_sr=sample_rate, target_sr=MODEL_SAMPLE_RATE)
    wav_tensor = torch.from_numpy(waveform).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        embedding = classifier.encode_batch(wav_tensor)
    return embedding.squeeze().cpu().numpy()

chapters = []
embeddings = []
for chapter, idx in tqdm(best_idx_by_chapter.items(), desc="Embedding chapters"):
    audio = dataset[idx][AUDIO_COLUMN]
    try:
        emb = embed_waveform(audio["array"], audio["sampling_rate"])
    except Exception as e:
        print(f"Skipping {chapter}: {e}", file=sys.stderr)
        continue
    chapters.append(chapter)
    embeddings.append(emb)

print(f"Extracted {len(embeddings)} chapter embeddings.")

## Step 4 — Cluster chapters into speakers

In [ ]:
if N_CLUSTERS:
    clustering = AgglomerativeClustering(n_clusters=N_CLUSTERS, metric="cosine", linkage="average")
else:
    clustering = AgglomerativeClustering(
        n_clusters=None, distance_threshold=DISTANCE_THRESHOLD, metric="cosine", linkage="average"
    )
labels = clustering.fit_predict(np.stack(embeddings))
speaker_map = {chapter: f"SPEAKER_{label:02d}" for chapter, label in zip(chapters, labels)}

print(f"Found {len(set(speaker_map.values()))} speaker(s) across {len(chapters)} chapters.")

### Sanity-check table: total duration / segments / chapters per speaker
**Read this before pushing.** A large number of tiny (1–2 chapter) clusters usually means over-clustering (embedding noise), not real distinct narrators — if you see that, raise `DISTANCE_THRESHOLD` above and re-run just the two cells above (embeddings don't need to be recomputed).

In [ ]:
durations = defaultdict(float)
seg_counts = defaultdict(int)
chapter_counts = defaultdict(set)
for chapter, duration in zip(dataset["chapter"], dataset["duration"]):
    sid = speaker_map.get(chapter, "UNKNOWN")
    durations[sid] += duration
    seg_counts[sid] += 1
    chapter_counts[sid].add(chapter)

for sid, secs in sorted(durations.items(), key=lambda x: x[1], reverse=True):
    print(f"{sid}\t{secs / 3600:.2f}h\t{seg_counts[sid]} segments\t{len(chapter_counts[sid])} chapters")

## Step 5 — Apply `speaker_id`, rename `audio_file` → `audio`, preview

In [ ]:
dataset = dataset.map(
    lambda example: {"speaker_id": speaker_map.get(example["chapter"], "UNKNOWN")},
    desc="speaker_id",
)
dataset = dataset.rename_column(AUDIO_COLUMN, "audio")

print(dataset)
print(dataset[0])

## Step 6 — Push to the Hub
Set `PUSH_TO_HUB = True` in the config cell above once you've checked the preview and the per-speaker table. This **replaces** the target split's data — not an incremental patch.

In [ ]:
if PUSH_TO_HUB:
    target_repo = OUTPUT_REPO_ID or REPO_ID
    dataset.push_to_hub(target_repo, split=SPLIT, private=True, token=HF_TOKEN)
    print(f"Pushed to https://huggingface.co/datasets/{target_repo}")
else:
    print("PUSH_TO_HUB is False — skipping upload. Flip it once you've sanity-checked everything above.")